# Stufe 1: Propositionsextraktion aus Literaturgeschichten

Dieses Notebook implementiert **Ebene 0, Stufe 1** der Annotationsrichtlinien:
Die Zerlegung literaturgeschichtlicher Sätze in Propositionen samt ihrer **Intra-Satz-Relationen**.

**Neu in dieser Version:** Automatische Erkennung der **Kapitelstruktur** aus dem PDF.
Jede Proposition wird ihrem Kapitel zugeordnet, sodass die nachfolgende Netzwerkanalyse
kapitelweise durchgeführt werden kann.

**Input:** Ein PDF einer Literaturgeschichte
**Output:**
- Ein DataFrame mit einer Zeile pro Proposition (inkl. Kapitelzuordnung)
- Export als CSV, XLSX und JSON

## 1. Setup und Konfiguration

In [1]:
# Installation (falls nötig)
# !pip install pdfplumber anthropic pandas openpyxl tqdm


In [2]:
import os
import pdfplumber
import pandas as pd
import json
import re
import time
from pathlib import Path
from anthropic import Anthropic
from tqdm.notebook import tqdm
from collections import Counter, OrderedDict


In [3]:
# ===== KONFIGURATION =====

# Anthropic API-Key
ANTHROPIC_API_KEY = os.getenv('MY_ANTHROPIC')

# Pfad zum PDF
PDF_PATH = "Beutin_complete_Direktzitat.pdf"  # <-- Hier anpassen

# Kapitelstruktur: Liste von (Kapiteltitel, erste Seite im PDF)
# Die Seitenzahlen beziehen sich auf die PDF-Seitennummern (1-basiert).
# Die Kapitel werden aus dem Inhaltsverzeichnis übernommen.
# Alternativ: CHAPTERS = None  → keine Kapitelstruktur
CHAPTERS = [
    ("Mittelalterliche Literatur", 11),
    ("Humanismus und Reformation", 69),
    ("Literatur des Barock", 113),
    ("Aufklärung", 161),
    ("Kunstepoche", 195),
    ("Vormärz", 251),
    ("Realismus und Gründerzeit", 305),
    ("Die literarische Moderne (1890–1920)", 355),
    ("Literatur in der Weimarer Republik", 401),
    ("Literatur im Dritten Reich", 447),
    ("Die deutsche Literatur des Exils", 465),
    ("Deutsche Literatur nach 1945", 493),
    ("Die Literatur der DDR", 525),
    ("Die Literatur der Bundesrepublik", 595),
    ("Tendenzen in der deutschsprachigen Gegenwartsliteratur seit 1989", 679),
]

# Seitenbereich (None = alle Seiten mit Kapitelzuordnung)
PAGE_START = None
PAGE_END = None

MODEL = "claude-opus-4-8"
BATCH_SIZE = 3
API_DELAY = 1.0

STEM = Path(PDF_PATH).stem
OUTPUT_CSV = f"{STEM}_stufe1.csv"
OUTPUT_XLSX = f"{STEM}_stufe1.xlsx"
OUTPUT_JSON = f"{STEM}_stufe1.json"
# Checkpoint-Konfiguration
CHECKPOINT_DIR = f"{STEM}_checkpoints"
CHECKPOINT_INTERVAL = 10  # Alle N Batches speichern


In [4]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

try:
    test = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Antworte nur mit: OK"}],
    )
    print(f"Verbindung OK. Modell: {MODEL}")
    print(f"Antwort: {test.content[0].text}")
except Exception as e:
    print(f"Fehler: {e}")


Verbindung OK. Modell: claude-opus-4-8
Antwort: OK


## 2. PDF-Textextraktion mit Kapitelstruktur

Die Kapitelgrenzen werden aus der CHAPTERS-Konfiguration übernommen.
Jeder Satz erhält eine Kapitelzuordnung basierend auf seiner Seitennummer.

In [5]:
def extract_clean_page_text(page, pg_idx):
    """Extrahiert bereinigten Text von einer Seite.
    Entfernt:
    - Seitenkopfzeilen (y < 35)
    - Marginalien (erkannt durch L\u00fcckenanalyse zwischen Rand- und Haupttext)
    - Bildunterschriften (kurzer Text direkt unter Bildern)
    """
    from collections import defaultdict
    
    words = page.extract_words(x_tolerance=3, y_tolerance=3)
    if not words:
        return ""
    
    # Zeilen nach y-Position gruppieren
    lines = defaultdict(list)
    for w in words:
        y_key = round(w['top'] / 11.5) * 11.5
        lines[y_key].append(w)
    
    clean_lines = []
    for y_key in sorted(lines.keys()):
        line_words = sorted(lines[y_key], key=lambda w: w['x0'])
        
        # Header \u00fcberspringen (oberer Seitenrand)
        if y_key < 35:
            continue
        
        # Marginalien per L\u00fcckenanalyse erkennen und entfernen
        main_words = line_words
        if len(line_words) > 1:
            for j in range(len(line_words) - 1):
                gap = line_words[j+1]['x0'] - line_words[j]['x1']
                if gap > 15:
                    left = line_words[:j+1]
                    right = line_words[j+1:]
                    left_w = left[-1]['x1'] - left[0]['x0']
                    right_w = right[-1]['x1'] - right[0]['x0']
                    main_words = right if left_w < right_w else left
                    break
        
        line_text = ' '.join(w['text'] for w in main_words)
        if line_text.strip():
            clean_lines.append(line_text)
    
    return '\n'.join(clean_lines)


def extract_pages_with_chapters(pdf_path, chapters=None, page_start=None, page_end=None):
    """Extrahiert bereinigten Text seitenweise mit Kapitelzuordnung.
    
    Entfernt automatisch Seitenkopfzeilen, Marginalien und Bildunterschriften.
    Seiten vor dem ersten Kapitel werden \u00fcbersprungen.
    """
    effective_start = page_start
    if chapters and not effective_start:
        effective_start = chapters[0][1]
        print(f"Beginne Extraktion bei PDF-Seite {effective_start} "
              f"(erstes Kapitel: '{chapters[0][0]}')")
    
    pages_data = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_num = i + 1
            if effective_start and page_num < effective_start:
                continue
            if page_end and page_num > page_end:
                break
            
            # Bereinigten Text extrahieren
            text = extract_clean_page_text(page, i)
            if not text or len(text.strip()) < 20:
                continue
            
            # Kapitel bestimmen
            chapter = "(Ohne Kapitel)"
            if chapters:
                for j, (title, start_page) in enumerate(chapters):
                    end_page = chapters[j + 1][1] if j + 1 < len(chapters) else 99999
                    if start_page <= page_num < end_page:
                        chapter = title
                        break
            
            pages_data.append({
                'page_num': page_num,
                'chapter': chapter,
                'text': text,
            })
    
    print(f"Extrahiert: {len(pages_data)} Seiten (bereinigt)")
    if chapters:
        ch_counts = Counter(p['chapter'] for p in pages_data)
        if '(Ohne Kapitel)' in ch_counts:
            print(f"  WARNUNG: {ch_counts['(Ohne Kapitel)']} Seiten ohne Kapitelzuordnung")
        print(f"\nKapitelverteilung:")
        for ch, count in ch_counts.items():
            print(f"  {ch}: {count} Seiten")
    return pages_data


In [6]:
def segment_sentences(text):
    """Segmentiert Text in Sätze. Robust gegenüber Initialen, Abkürzungen, Ordinalzahlen."""
    # Silbentrennung zusammenführen
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    protected = text
    PH = '__DOT__'

    # 1a) Namensinitialen
    protected = re.sub(
        r'\b([A-Z\u00c4\u00d6\u00dca-z\u00e4\u00f6\u00fc])\.(?=\s+[A-Z\u00c4\u00d6\u00dca-z\u00e4\u00f6\u00fc])',
        r'\g<1>' + PH, protected
    )
    # 1b) Ordinalzahlen
    protected = re.sub(
        r'(\d)\.(?=\s+[A-Z\u00c4\u00d6\u00dca-z\u00e4\u00f6\u00fc])',
        r'\g<1>' + PH, protected
    )
    # 1c) Einfache Abkürzungen
    for a in ['bzw','vgl','sog','ca','Dr','Prof','St','Bd','Jh','Jhs',
              'Anm','Kap','Aufl','Hrsg','ebd','ders','dies','Nr','Abs',
              'Abt','usw','etc','geb','gest','Abb','Tab','Verf']:
        protected = re.sub(r'\b' + a + r'\.', a + PH, protected)
    # 1d) Mehrteilige Abkürzungen
    for pat, repl in [
        (r'z' + PH + r'\s*B\.', 'z' + PH + 'B' + PH),
        (r'z\.\s*B\.', 'z' + PH + 'B' + PH),
        (r'd' + PH + r'\s*h\.', 'd' + PH + 'h' + PH),
        (r'd\.\s*h\.', 'd' + PH + 'h' + PH),
        (r'u' + PH + r'\s*a\.', 'u' + PH + 'a' + PH),
        (r'u\.\s*a\.', 'u' + PH + 'a' + PH),
        (r'v' + PH + r'\s*a\.', 'v' + PH + 'a' + PH),
        (r'v\.\s*a\.', 'v' + PH + 'a' + PH),
    ]:
        protected = re.sub(pat, repl, protected)
    # 1e) S. + Zahl
    protected = re.sub(r'\bS\.(?=\s*\d)', 'S' + PH, protected)
    # 1f) Anführungszeichen/Klammern vor Punkt
    protected = re.sub(
        r'([\u00bb\u00ab\u203a\u2039\u201e\u201c\u201d\u2018\u2019\u201a\)\]])\.',
        r'\g<1>' + PH, protected
    )

    # Satzsegmentierung
    raw = re.split(
        r'(?<=[.!?])\s+(?=[A-Z\u00c4\u00d6\u00dc\u201e\u00bb\u203a\[\(])',
        protected
    )
    sentences = [s.replace(PH, '.').strip() for s in raw if s.strip()]
    # Zu kurze Fragmente an vorherigen Satz anhängen
    merged = []
    for s in sentences:
        if merged and len(s) < 30 and not s[0].isupper():
            merged[-1] += ' ' + s
        else:
            merged.append(s)
    return merged

In [7]:
def extract_sentences_with_chapters(pages_data):
    """Segmentiert Text in S\u00e4tze mit Kapitelzuordnung.
    
    WICHTIG: Der Text wird KAPITELWEISE zusammengefuehrt, bevor die
    Satzsegmentierung erfolgt. Dadurch werden Suetze, die ueber
    Seitengrenzen hinweg laufen, korrekt zusammengefuegt.
    
    Returns:
        List of tuples: [(satz_id, text, chapter), ...]
    """
    from collections import OrderedDict
    
    # Schritt 1: Text kapitelweise zusammenf\u00fchren
    chapter_texts = OrderedDict()
    for page in pages_data:
        chapter = page['chapter']
        text = page['text']
        if chapter not in chapter_texts:
            chapter_texts[chapter] = []
        chapter_texts[chapter].append(text)
    
    # Seiten eines Kapitels mit Leerzeichen verbinden
    # (nicht mit Newline — segment_sentences behandelt Newlines intern)
    for chapter in chapter_texts:
        chapter_texts[chapter] = ' '.join(chapter_texts[chapter])
    
    print(f"Kapitel zusammengef\u00fchrt:")
    for ch, text in chapter_texts.items():
        print(f"  {ch[:50]}: {len(text)} Zeichen")
    
    # Schritt 2: Kapitelweise segmentieren
    all_sentences = []
    sentence_counter = 0
    
    for chapter, full_text in chapter_texts.items():
        sentences = segment_sentences(full_text)
        for s in sentences:
            sentence_counter += 1
            sid = f"S-{sentence_counter:03d}"
            all_sentences.append((sid, s, chapter))
    
    print(f"\nSegmentiert: {len(all_sentences)} S\u00e4tze")
    ch_counts = Counter(ch for _, _, ch in all_sentences)
    print(f"\nS\u00e4tze pro Kapitel:")
    for ch, count in ch_counts.items():
        print(f"  {ch}: {count}")
    return all_sentences


In [8]:
pages_data = extract_pages_with_chapters(PDF_PATH, CHAPTERS, PAGE_START, PAGE_END)
indexed_sentences = extract_sentences_with_chapters(pages_data)

print(f"\nBeispiele:")
for sid, text, ch in indexed_sentences[:5]:
    preview = text[:100] + '...' if len(text) > 100 else text
    print(f"  {sid} [{ch[:30]}]: {preview}")

Beginne Extraktion bei PDF-Seite 11 (erstes Kapitel: 'Mittelalterliche Literatur')
Extrahiert: 793 Seiten (bereinigt)

Kapitelverteilung:
  Mittelalterliche Literatur: 58 Seiten
  Humanismus und Reformation: 44 Seiten
  Literatur des Barock: 48 Seiten
  Aufklärung: 34 Seiten
  Kunstepoche: 56 Seiten
  Vormärz: 54 Seiten
  Realismus und Gründerzeit: 50 Seiten
  Die literarische Moderne (1890–1920): 46 Seiten
  Literatur in der Weimarer Republik: 46 Seiten
  Literatur im Dritten Reich: 18 Seiten
  Die deutsche Literatur des Exils: 28 Seiten
  Deutsche Literatur nach 1945: 32 Seiten
  Die Literatur der DDR: 70 Seiten
  Die Literatur der Bundesrepublik: 84 Seiten
  Tendenzen in der deutschsprachigen Gegenwartsliteratur seit 1989: 125 Seiten
Kapitel zusammengeführt:
  Mittelalterliche Literatur: 170762 Zeichen
  Humanismus und Reformation: 154682 Zeichen
  Literatur des Barock: 152632 Zeichen
  Aufklärung: 107600 Zeichen
  Kunstepoche: 187621 Zeichen
  Vormärz: 180334 Zeichen
  Realismus un

## 3. Propositionsextraktion via Claude (Chain of Thought)

In [9]:
# System-Prompt: CoT-Fassung
# Option 1: Aus Datei laden (Prompt kann unabhängig aktualisiert werden)
# Option 2: Direkt als String einbetten (Fallback)

PROMPT_FILE = 'prompt_intra-satz_cot.md'

if os.path.exists(PROMPT_FILE):
    raw = open(PROMPT_FILE, 'r', encoding='utf-8').read()
    # Wenn die Datei mit 'SYSTEM_PROMPT = """' beginnt, den Python-Wrapper entfernen
    if raw.strip().startswith('SYSTEM_PROMPT'):
        # Extrahiere den String zwischen den dreifachen Anführungszeichen
        start = raw.index('"""') + 3
        end = raw.rindex('"""')
        SYSTEM_PROMPT = raw[start:end]
    else:
        SYSTEM_PROMPT = raw
    print(f"System-Prompt aus Datei geladen: {len(SYSTEM_PROMPT)} Zeichen")
else:
    print(f"WARNUNG: {PROMPT_FILE} nicht gefunden!")
    print("Bitte den Prompt unten als String einsetzen oder die Datei bereitstellen.")
    SYSTEM_PROMPT = "PLACEHOLDER"

print(f"Prompt-Anfang: {SYSTEM_PROMPT[:100]}...")

System-Prompt aus Datei geladen: 7659 Zeichen
Prompt-Anfang: Du bist ein Annotationswerkzeug für die inferentielle Analyse von Literaturgeschichten nach Robert B...


In [10]:
def build_user_prompt(batch):
    """batch = list of (sid, text, chapter)"""
    lines = [f"{sid}: {text}" for sid, text, _ in batch]
    satz_block = "\n\n".join(lines)
    return (
        f"Extrahiere die Propositionen und Intra-Satz-Relationen:\n\n"
        f"{satz_block}\n\n"
        f"WICHTIG: Antworte NUR mit dem JSON-Array. "
        f"Beginne direkt mit [ und ende mit ]. "
        f"Kein Text davor, kein Markdown, keine Erkl\u00e4rungen."
    )


def call_api(batch, max_retries=3):
    """Ruft die Anthropic-API auf und gibt die Ergebnisse zur\u00fcck.
    
    Bei Fehlern werden fehlgeschlagene S\u00e4tze mit _failed=True markiert,
    damit sie sp\u00e4ter erneut versucht werden k\u00f6nnen.
    """
    prompt = build_user_prompt(batch)
    batch_sids = {sid for sid, _, _ in batch}
    
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=8192,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            
            # Anthropic SDK: content ist eine Liste von ContentBlocks
            raw = resp.content[0].text.strip() if resp.content else ""
            
            if not raw:
                tqdm.write(f"  WARNUNG: Leere API-Antwort (Versuch {attempt+1}/{max_retries})")
                if attempt < max_retries - 1: time.sleep(5)
                continue
            
            # Stop-Reason pr\u00fcfen
            if resp.stop_reason == 'max_tokens':
                tqdm.write(f"  WARNUNG: Antwort bei max_tokens abgeschnitten (Versuch {attempt+1})")
            
            if raw.startswith("```"):
                raw = re.sub(r'^```(?:json)?\s*', '', raw)
                raw = re.sub(r'\s*```$', '', raw)
            results = json.loads(raw)
            if not isinstance(results, list):
                results = [results]
            
            # Pr\u00fcfe, ob alle Batch-S\u00e4tze in der Antwort enthalten sind
            returned_sids = {r.get('satz_id', '') for r in results}
            missing_sids = batch_sids - returned_sids
            if missing_sids:
                tqdm.write(f"  WARNUNG: {len(missing_sids)} S\u00e4tze fehlen in API-Antwort: {missing_sids}")
                for sid in missing_sids:
                    results.append({"satz_id": sid, "propositionen": [],
                                    "intra_relationen": [], "_failed": True})
            
            return results
        except json.JSONDecodeError as e:
            tqdm.write(f"  JSON-Fehler (Versuch {attempt+1}/{max_retries}): {e}")
            tqdm.write(f"    Raw-Antwort ({len(raw)} Zeichen): {raw[:200]}")
            if attempt < max_retries - 1: time.sleep(2)
        except Exception as e:
            error_type = type(e).__name__
            tqdm.write(f"  {error_type} (Versuch {attempt+1}/{max_retries}): {e}")
            # Bei Rate-Limits l\u00e4nger warten
            if 'rate' in str(e).lower() or 'overloaded' in str(e).lower():
                wait = 30 if attempt == 0 else 60
                tqdm.write(f"    Rate-Limit/Overloaded — warte {wait}s")
                time.sleep(wait)
            elif attempt < max_retries - 1:
                time.sleep(5)
    
    tqdm.write(f"  FEHLER: Batch {[sid for sid,_,_ in batch]} nach {max_retries} Versuchen gescheitert")
    return [{"satz_id": sid, "propositionen": [], "intra_relationen": [],
             "_failed": True} for sid, _, _ in batch]


def save_checkpoint(all_results, batch_idx, checkpoint_dir):
    os.makedirs(checkpoint_dir, exist_ok=True)
    cp_path = os.path.join(checkpoint_dir, f"checkpoint_batch_{batch_idx:04d}.json")
    with open(cp_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.json")
    with open(latest_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    return cp_path


def load_checkpoint(checkpoint_dir):
    latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.json")
    if os.path.exists(latest_path):
        with open(latest_path, 'r', encoding='utf-8') as f:
            results = json.load(f)
        processed_sids = {r.get('satz_id', '') for r in results
                          if not r.get('_failed', False)}
        n_failed = sum(1 for r in results if r.get('_failed', False))
        results = [r for r in results if not r.get('_failed', False)]
        print(f"Checkpoint geladen: {len(results)} S\u00e4tze erfolgreich verarbeitet")
        if n_failed > 0:
            print(f"  {n_failed} fehlgeschlagene S\u00e4tze werden erneut versucht")
        return results, processed_sids
    return [], set()


def process_all(indexed_sentences, batch_size=BATCH_SIZE,
                checkpoint_dir=CHECKPOINT_DIR,
                checkpoint_interval=CHECKPOINT_INTERVAL,
                resume=True):
    """Verarbeitet alle S\u00e4tze mit Checkpoint-Support und Retry-Logik."""
    all_results = []
    processed_sids = set()
    if resume:
        all_results, processed_sids = load_checkpoint(checkpoint_dir)
    
    remaining = [(sid, text, ch) for sid, text, ch in indexed_sentences
                 if sid not in processed_sids]
    
    if not remaining:
        print("Alle S\u00e4tze bereits verarbeitet (Checkpoint komplett).")
        return all_results
    
    print(f"Zu verarbeiten: {len(remaining)} von {len(indexed_sentences)} S\u00e4tzen")
    if processed_sids:
        print(f"  ({len(processed_sids)} aus Checkpoint \u00fcbernommen)")
    
    batches = [remaining[i:i+batch_size]
               for i in range(0, len(remaining), batch_size)]
    
    for batch_idx, batch in enumerate(tqdm(batches, desc="Stufe 1: Propositionsextraktion")):
        results = call_api(batch)
        meta_lookup = {sid: (text, ch) for sid, text, ch in batch}
        for r in results:
            sid = r.get('satz_id', '')
            text, ch = meta_lookup.get(sid, ('', '(Unbekannt)'))
            r['kapitel'] = ch
            r['originaltext'] = text
        all_results.extend(results)
        
        if (batch_idx + 1) % checkpoint_interval == 0:
            cp_path = save_checkpoint(all_results, batch_idx, checkpoint_dir)
            tqdm.write(f"  Checkpoint: {len(all_results)} S\u00e4tze \u2192 {cp_path}")
        
        time.sleep(API_DELAY)
    
    n_failed = sum(1 for r in all_results if r.get('_failed', False))
    n_success = len(all_results) - n_failed
    n_with_props = sum(1 for r in all_results
                       if r.get('propositionen') and not r.get('_failed', False))
    
    save_checkpoint(all_results, len(batches), checkpoint_dir)
    print(f"Finaler Checkpoint: {len(all_results)} S\u00e4tze")
    print(f"  Erfolgreich: {n_success}")
    print(f"  Davon mit Propositionen: {n_with_props}")
    if n_failed > 0:
        print(f"  FEHLGESCHLAGEN: {n_failed} (werden bei erneutem Lauf mit resume=True erneut versucht)")
    
    return all_results


In [ ]:
# Verarbeitung starten (setzt automatisch vom letzten Checkpoint fort)
# Bei resume=False wird von vorne begonnen und bestehende Checkpoints ignoriert.
results = process_all(indexed_sentences, resume=True)
print(f"Verarbeitet: {len(results)} Sätze")


Checkpoint geladen: 11540 Sätze erfolgreich verarbeitet
  70 fehlgeschlagene Sätze werden erneut versucht
Zu verarbeiten: 70 von 11606 Sätzen
  (11539 aus Checkpoint übernommen)


Stufe 1: Propositionsextraktion:   0%|          | 0/24 [00:00<?, ?it/s]

  WARNUNG: 3 Sätze fehlen in API-Antwort: {'S-11109', 'S-11110', 'S-11108'}


In [ ]:
# === Postprocessing: Konsistenzprüfung relationstyp vs. intra_relationen ===
# Stellt sicher, dass jede Proposition mit relationstyp != PR-UNA
# tatsächlich in einem Paar im intra_relationen-Array vorkommt.

def postprocess_consistency(results):
    """Korrigiert Inkonsistenzen zwischen relationstyp und intra_relationen."""
    n_fixed = 0
    for item in results:
        rels = item.get('intra_relationen', [])
        # IDs aller Propositionen, die in Paaren vorkommen
        paired_ids = set()
        for r in rels:
            paired_ids.add(r.get('von', ''))
            paired_ids.add(r.get('nach', ''))
        
        for prop in item.get('propositionen', []):
            pid = prop.get('id', prop.get('prop_id', ''))
            reltyp = prop.get('relationstyp', 'PR-UNA')
            
            # Fall 1: relationstyp != PR-UNA, aber nicht in Paaren
            if reltyp in ('PR-BER', 'PR-FLG', 'PR-INK') and pid not in paired_ids:
                prop['relationstyp'] = 'PR-UNA'
                n_fixed += 1
            
            # Fall 2: relationstyp == PR-UNA, aber in Paaren
            elif reltyp == 'PR-UNA' and pid in paired_ids:
                # Bestimme den tatsächlichen Typ aus den Paaren
                pair_types = set()
                for r in rels:
                    if r.get('von') == pid or r.get('nach') == pid:
                        pair_types.add(r.get('typ', ''))
                if pair_types:
                    prop['relationstyp'] = ', '.join(sorted(pair_types))
                    n_fixed += 1
    
    if n_fixed > 0:
        print(f"Konsistenzprüfung: {n_fixed} Propositionen korrigiert")
    else:
        print("Konsistenzprüfung: Keine Inkonsistenzen gefunden")
    return results

results = postprocess_consistency(results)


In [ ]:
# === Manueller Checkpoint-Recovery ===
# Falls das Notebook abstürzt oder die Verbindung abbricht,
# kann hier der letzte Checkpoint manuell geladen werden:

# from pathlib import Path
# cp_dir = f"{Path(PDF_PATH).stem}_checkpoints"
# cp_file = os.path.join(cp_dir, "checkpoint_latest.json")
# if os.path.exists(cp_file):
#     with open(cp_file, 'r', encoding='utf-8') as f:
#         results = json.load(f)
#     print(f"Checkpoint manuell geladen: {len(results)} Sätze")
# else:
#     print(f"Kein Checkpoint gefunden in {cp_dir}")


## 4. DataFrame erstellen

In [ ]:
def _get_field(d, *keys, default=""):
    if not isinstance(d, dict): return default
    for k in keys:
        if k in d and d[k] not in (None, ""): return d[k]
    lower_d = {k.lower(): v for k, v in d.items()}
    for k in keys:
        if k.lower() in lower_d and lower_d[k.lower()] not in (None, ""): return lower_d[k.lower()]
    return default


def _get_relations(item):
    for key in ["intra_relationen", "relationen", "intra_relations", "relations"]:
        if key in item and isinstance(item[key], list): return item[key]
    return []


def _normalize_relation(rel):
    return {
        "von": _get_field(rel, "von", "from", "source"),
        "nach": _get_field(rel, "nach", "to", "target"),
        "typ": _get_field(rel, "typ", "type", "relationstyp"),
        "konnektor": _get_field(rel, "konnektor", "connector", "marker", default=""),
        "reasoning": _get_field(rel, "reasoning", default=""),
    }


def results_to_dataframe(results, indexed_sentences=None):
    """Konvertiert API-Ergebnisse in einen DataFrame.
    
    Originaltext wird prim\u00e4r aus den results selbst gelesen (Checkpoint-kompatibel).
    Fallback auf indexed_sentences, falls vorhanden.
    Fragmente und \u00dcberschriften (Zeilen ohne Propositionen) werden automatisch entfernt.
    """
    orig_lookup = {}
    if indexed_sentences:
        orig_lookup = {sid: text for sid, text, _ in indexed_sentences}
    
    rows = []
    n_fragments = 0
    for item in results:
        satz_id = item.get("satz_id", item.get("sentence_id", ""))
        # Originaltext: zuerst aus result, dann Fallback auf indexed_sentences
        originaltext = item.get("originaltext", orig_lookup.get(satz_id, ""))
        kapitel = item.get("kapitel", "")
        props = item.get("propositionen", item.get("propositions", []))
        raw_rels = _get_relations(item)
        rels = [_normalize_relation(r) for r in raw_rels]

        # Fragmente/\u00dcberschriften \u00fcberspringen (nicht in DataFrame aufnehmen)
        if not props:
            n_fragments += 1
            continue

        for prop in props:
            pid = _get_field(prop, "id", "prop_id")
            reltyp_prop = _get_field(prop, "relationstyp", "relation_type", default="PR-UNA")
            prop_rels = [r for r in rels if r["von"] == pid or r["nach"] == pid]
            if prop_rels:
                rel_text = "; ".join(
                    f"{r['von']} \u2192 {r['nach']}: {r['typ']} | {r['konnektor']}"
                    for r in prop_rels
                )
                typen_text = ", ".join(sorted(set(r["typ"] for r in prop_rels)))
                reasoning_text = " | ".join(r["reasoning"] for r in prop_rels if r["reasoning"])
            else:
                rel_text = ""
                typen_text = reltyp_prop
                reasoning_text = ""
            rows.append({"Kapitel": kapitel, "Satz-ID": satz_id,
                         "Originaltext": originaltext,
                         "Propositions-ID": pid,
                         "Proposition": _get_field(prop, "text", "proposition"),
                         "Intra-Satz-Relationen": rel_text,
                         "Relationstypen": typen_text,
                         "Reasoning": reasoning_text})
    
    if n_fragments > 0:
        print(f"Fragmente/\u00dcberschriften entfernt: {n_fragments}")
    
    return pd.DataFrame(rows)


In [ ]:
# DataFrame erstellen (Fragmente/\u00dcberschriften werden automatisch entfernt)
df = results_to_dataframe(results, indexed_sentences)
n_saetze = df['Satz-ID'].nunique()
n_kapitel = df['Kapitel'].nunique() if 'Kapitel' in df.columns else 0
print(f"DataFrame: {len(df)} Propositionen, {n_saetze} S\u00e4tze, {n_kapitel} Kapitel")
print(f"Spalten: {list(df.columns)}")


In [ ]:
# Diagnose: Welche Schlüssel verwendet die API tatsächlich?
from collections import Counter

top_keys = Counter()
rel_keys = Counter()
prop_keys = Counter()
for item in results:
    top_keys.update(item.keys())
    for p in item.get('propositionen', item.get('propositions', [])):
        if isinstance(p, dict):
            prop_keys.update(p.keys())
    for key in ['intra_relationen', 'relationen', 'intra_relations', 'relations']:
        if key in item:
            for rel in item.get(key, []):
                if isinstance(rel, dict):
                    rel_keys.update(rel.keys())
            break

print("Top-Level-Schlüssel in API-Antworten:")
for k, c in top_keys.most_common():
    print(f"  {k}: {c}x")
print("\nPropositions-Schlüssel:")
for k, c in prop_keys.most_common():
    print(f"  {k}: {c}x")
print("\nRelationen-Schlüssel:")
for k, c in rel_keys.most_common():
    print(f"  {k}: {c}x")

In [ ]:
pd.set_option('display.max_colwidth', 100)
df.head(15)

## 5. Export

In [ ]:
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"CSV: {OUTPUT_CSV}")

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Propositionen')
    ws = writer.sheets['Propositionen']
    from openpyxl.styles import Font, PatternFill, Alignment
    hf = PatternFill(start_color='2B4C7E', end_color='2B4C7E', fill_type='solid')
    for cell in ws[1]:
        cell.fill = hf
        cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = Font(name='Arial', size=9)
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    # Spaltenbreiten
    col_widths = {'A': 30, 'B': 8, 'C': 55, 'D': 12, 'E': 55,
                  'F': 45, 'G': 16, 'H': 60}
    for col, w in col_widths.items():
        ws.column_dimensions[col].width = w
    ws.auto_filter.ref = ws.dimensions
print(f"XLSX: {OUTPUT_XLSX}")

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"JSON: {OUTPUT_JSON}")

## 6. Statistiken

In [ ]:
# Statistiken (Fragmente/\u00dcberschriften sind bereits entfernt)
valid_results = [r for r in results if r.get('propositionen')]
total_sents = len(valid_results)
total_props = sum(len(r.get('propositionen', [])) for r in valid_results)
total_rels = sum(len(_get_relations(r)) for r in valid_results)
multi_prop = sum(1 for r in valid_results if len(r.get('propositionen', [])) > 1)
n_fragments = sum(1 for r in results if not r.get('propositionen'))

print(f"Gesamtstatistik")
print(f"{'='*50}")
print(f"S\u00e4tze gesamt (ohne Fragmente): {total_sents}")
print(f"Propositionen gesamt:          {total_props}")
print(f"Intra-Relationen gesamt:       {total_rels}")
print(f"S\u00e4tze mit >1 Proposition:      {multi_prop} ({multi_prop/max(total_sents,1)*100:.1f}%)")
print(f"Entfernte Fragmente/\u00dcberschr.: {n_fragments}")
print(f"\u00d8 Propositionen/Satz:            {total_props/max(total_sents,1):.2f}")

# Relationstyp-Verteilung
rel_types = Counter()
for r in valid_results:
    for rel in _get_relations(r):
        rel_types[_normalize_relation(rel)['typ']] += 1
if rel_types:
    print(f"\nRelationstyp-Verteilung:")
    for typ, count in rel_types.most_common():
        print(f"  {typ}: {count} ({count/max(total_rels,1)*100:.1f}%)")

# Pro Kapitel
if CHAPTERS:
    print(f"\nPro Kapitel:")
    for ch in dict.fromkeys(r.get('kapitel','') for r in valid_results):
        ch_results = [r for r in valid_results if r.get('kapitel') == ch]
        ch_props = sum(len(r.get('propositionen', [])) for r in ch_results)
        ch_rels = sum(len(_get_relations(r)) for r in ch_results)
        print(f"  {ch[:50]}: {len(ch_results)} S\u00e4tze, {ch_props} Props, {ch_rels} Rels")
